# Notebook 05 — Integrated Decision System and Deployment Governance

This is the final notebook in the five-notebook Nordic Wind Decision System.

It converts the forecasting and commercial-evaluation results into:

- conservative, balanced, and profit-seeking operating modes;
- area- and horizon-specific policy assignments;
- production, pilot, challenger, and shadow evidence tiers;
- fee and price-spread stress tests;
- automated fallback rules;
- monitoring thresholds;
- a final deployment configuration for operational implementation.

The notebook does **not** retrain forecasting models. It consumes the audited
artifacts from Notebooks 03 and 04 and freezes the final decision layer.


## Why the exact track has one probabilistic/commercial evaluation fold

Notebook 02 contains **two exact-run deterministic outer folds**. The exact
windows were shortened to:

- 90 minimum training days;
- 21 validation days;
- 42 test days;
- 42-day step.

That change successfully created two deterministic test folds.

Notebook 03 then required a strictly earlier out-of-sample residual history to
calibrate each probabilistic distribution and construct joint scenarios.
Consequently:

- exact fold 1 became calibration history;
- exact fold 2 became the first leakage-safe probabilistic evaluation fold.

The scenario loop deliberately evaluates `fold_ids[1:]`. This is why Notebook
04 observes one exact commercial fold even though Notebook 02 evaluated two
deterministic exact folds.

Obtaining two exact probabilistic folds requires either:

1. at least three deterministic exact folds; or
2. an independent pre-fold calibration sample constructed before exact fold 1.

Notebook 05 therefore classifies all exact-track commercial conclusions as
**shadow evidence**, not production evidence.


## Why no commercial comparison passed the 5% statistical tests

The commercial point estimates can be economically large without being
statistically precise.

The tests correctly account for:

- serial dependence through seven-day blocks and HAC standard errors;
- evaluation at the daily rather than hourly independence level;
- highly volatile and heavy-tailed imbalance-price spreads;
- six alternative nomination policies within each context;
- Holm multiple-testing adjustment;
- only 180 D2 days and 40 exact days;
- only one exact commercial fold.

For the strongest D2 DK1 result, the raw online-profit policy had about a 90%
bootstrap probability of beating the point nomination, but its 95% confidence
interval still crossed zero. A 90% directional probability is promising
evidence; it is not conventional two-sided 5% significance.

This notebook therefore separates:

- **economic relevance**;
- **pilot eligibility**;
- **production statistical proof**.

No policy is promoted to statistically proven production superiority merely
because its retrospective point estimate is large.


In [ ]:
# ============================================================
# 1. IMPORTS AND CONFIGURATION
# ============================================================

from __future__ import annotations

from pathlib import Path
from typing import Any
import hashlib
import json
import math
import platform
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display


RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

NOTEBOOK_03_OUTPUT_RUN = (
    "notebook_03_probabilistic"
)

NOTEBOOK_04_OUTPUT_RUN = (
    "notebook_04_commercial"
)

NOTEBOOK_05_OUTPUT_RUN = (
    "notebook_05_integrated_decision_system"
)

PRICE_AREAS = (
    "DK1",
    "DK2",
)

PANELS = (
    "d2",
    "exact",
)

OPERATING_MODES = (
    "conservative",
    "balanced",
    "profit_seeking",
)

# Evidence gates.
MINIMUM_PRODUCTION_FOLDS = 2
MINIMUM_PRODUCTION_DAYS = 120
MINIMUM_PILOT_BOOTSTRAP_PROBABILITY = 0.80
MINIMUM_PRODUCTION_BOOTSTRAP_PROBABILITY = 0.975
MAXIMUM_PILOT_IMBALANCE_INCREASE_PCT = 75.0
MAXIMUM_PRODUCTION_IMBALANCE_INCREASE_PCT = 25.0

# Statistical evidence extrapolation.
TWO_SIDED_95_CRITICAL_VALUE = 1.96
CONSERVATIVE_MULTIPLE_TEST_CRITICAL_VALUE = 2.60

# Operational monitoring.
ROLLING_COMMERCIAL_WINDOW_DAYS = 28
ROLLING_DATA_WINDOW_DAYS = 7
MINIMUM_MARKET_DATA_COMPLETENESS = 0.99
MINIMUM_ROLLING_COMMERCIAL_DELTA_EUR_PER_MWH = 0.0
MAXIMUM_ROLLING_IMBALANCE_INCREASE_PCT = 50.0
MAXIMUM_SELECTED_QUANTILE = 0.90
MINIMUM_SELECTED_QUANTILE = 0.10

print(
    {
        "python": sys.version.split()[0],
        "platform": platform.platform(),
        "operating_modes": OPERATING_MODES,
        "production_minimum_folds": (
            MINIMUM_PRODUCTION_FOLDS
        ),
        "production_minimum_days": (
            MINIMUM_PRODUCTION_DAYS
        ),
    }
)


In [ ]:
# ============================================================
# 2. PROJECT ROOT AND PATHS
# ============================================================

def find_project_root(
    starting_path: Path | None = None,
) -> Path:
    current = Path(
        starting_path or Path.cwd()
    ).resolve()

    for candidate in (
        current,
        *current.parents,
    ):
        if (
            candidate.joinpath(
                "pyproject.toml"
            ).exists()
            and candidate.joinpath(
                "data"
            ).exists()
        ):
            return candidate

    raise FileNotFoundError(
        "Could not locate the project root. "
        "Expected pyproject.toml and data/."
    )


PROJECT_ROOT = find_project_root()

NOTEBOOK_03_DIR = (
    PROJECT_ROOT
    / "outputs"
    / NOTEBOOK_03_OUTPUT_RUN
)

NOTEBOOK_04_DIR = (
    PROJECT_ROOT
    / "outputs"
    / NOTEBOOK_04_OUTPUT_RUN
)

NOTEBOOK_05_DIR = (
    PROJECT_ROOT
    / "outputs"
    / NOTEBOOK_05_OUTPUT_RUN
)

AUDIT_DIR = NOTEBOOK_05_DIR / "audits"
CONFIG_DIR = NOTEBOOK_05_DIR / "config"
FIGURE_DIR = NOTEBOOK_05_DIR / "figures"
MONITORING_DIR = NOTEBOOK_05_DIR / "monitoring"
TABLE_DIR = NOTEBOOK_05_DIR / "tables"

for directory in (
    AUDIT_DIR,
    CONFIG_DIR,
    FIGURE_DIR,
    MONITORING_DIR,
    TABLE_DIR,
):
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


NOTEBOOK_03_VERIFICATION_PATH = (
    NOTEBOOK_03_DIR
    / "audits"
    / "notebook_03_verification.json"
)

NOTEBOOK_03_CALIBRATION_MANIFEST_PATH = (
    NOTEBOOK_03_DIR
    / "audits"
    / "notebook_03_calibration_manifest.csv"
)

NOTEBOOK_04_VERIFICATION_PATH = (
    NOTEBOOK_04_DIR
    / "audits"
    / "notebook_04_verification.json"
)

SETTLEMENT_PATH = (
    NOTEBOOK_04_DIR
    / "settlement"
    / "notebook_04_settlement_hourly.parquet"
)

COMMERCIAL_SUMMARY_PATH = (
    NOTEBOOK_04_DIR
    / "tables"
    / "notebook_04_commercial_summary.csv"
)

FOLD_SUMMARY_PATH = (
    NOTEBOOK_04_DIR
    / "tables"
    / "notebook_04_commercial_summary_by_fold.csv"
)

COMPARISON_VS_POINT_PATH = (
    NOTEBOOK_04_DIR
    / "tables"
    / "notebook_04_comparison_vs_point.csv"
)

INFERENCE_VS_POINT_PATH = (
    NOTEBOOK_04_DIR
    / "tables"
    / "notebook_04_inference_vs_point.csv"
)

RAW_VS_GATED_PATH = (
    NOTEBOOK_04_DIR
    / "tables"
    / "notebook_04_raw_vs_gated_summary.csv"
)

PORTFOLIO_SUMMARY_PATH = (
    NOTEBOOK_04_DIR
    / "tables"
    / "notebook_04_denmark_portfolio_summary.csv"
)

SENSITIVITY_PATH = (
    NOTEBOOK_04_DIR
    / "tables"
    / "notebook_04_fee_and_spread_sensitivity.csv"
)

SHORTLIST_PATH = (
    NOTEBOOK_04_DIR
    / "tables"
    / "notebook_04_commercial_shortlist.csv"
)

print("Project root:", PROJECT_ROOT)
print("Notebook 04:", NOTEBOOK_04_DIR)
print("Notebook 05:", NOTEBOOK_05_DIR)


In [ ]:
# ============================================================
# 3. LOAD AND VERIFY NOTEBOOK 03–04 ARTIFACTS
# ============================================================

def sha256_file(
    path: Path,
    chunk_size: int = 1024 * 1024,
) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as file:
        while True:
            chunk = file.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


required_paths = (
    NOTEBOOK_03_VERIFICATION_PATH,
    NOTEBOOK_04_VERIFICATION_PATH,
    SETTLEMENT_PATH,
    COMMERCIAL_SUMMARY_PATH,
    FOLD_SUMMARY_PATH,
    COMPARISON_VS_POINT_PATH,
    INFERENCE_VS_POINT_PATH,
    RAW_VS_GATED_PATH,
    PORTFOLIO_SUMMARY_PATH,
    SENSITIVITY_PATH,
    SHORTLIST_PATH,
)


for required_path in required_paths:
    if not required_path.exists():
        raise FileNotFoundError(
            f"Required artifact not found: "
            f"{required_path}"
        )


with NOTEBOOK_03_VERIFICATION_PATH.open(
    "r",
    encoding="utf-8",
) as file:
    notebook_03_verification = (
        json.load(file)
    )


with NOTEBOOK_04_VERIFICATION_PATH.open(
    "r",
    encoding="utf-8",
) as file:
    notebook_04_verification = (
        json.load(file)
    )


if notebook_03_verification.get(
    "verification_status"
) != "PROBABILISTIC_MODELLING_COMPLETE":
    raise RuntimeError(
        "Notebook 03 is not verified complete."
    )


if notebook_04_verification.get(
    "verification_status"
) != "COMMERCIAL_EVALUATION_COMPLETE":
    raise RuntimeError(
        "Notebook 04 is not verified complete."
    )


if notebook_04_verification.get(
    "blocking_failures"
):
    raise RuntimeError(
        "Notebook 04 contains blocking failures."
    )


settlement = pd.read_parquet(
    SETTLEMENT_PATH
)

commercial_summary = pd.read_csv(
    COMMERCIAL_SUMMARY_PATH
)

commercial_fold_summary = pd.read_csv(
    FOLD_SUMMARY_PATH
)

comparison_vs_point = pd.read_csv(
    COMPARISON_VS_POINT_PATH
)

inference_vs_point = pd.read_csv(
    INFERENCE_VS_POINT_PATH
)

raw_vs_gated = pd.read_csv(
    RAW_VS_GATED_PATH
)

portfolio_summary = pd.read_csv(
    PORTFOLIO_SUMMARY_PATH
)

commercial_sensitivity = pd.read_csv(
    SENSITIVITY_PATH
)

commercial_shortlist = pd.read_csv(
    SHORTLIST_PATH
)


settlement[
    "timestamp_utc"
] = pd.to_datetime(
    settlement[
        "timestamp_utc"
    ],
    utc=True,
)

settlement[
    "delivery_date"
] = pd.to_datetime(
    settlement[
        "delivery_date"
    ]
).dt.date


input_manifest = {
    "notebook_03_status": (
        notebook_03_verification.get(
            "verification_status"
        )
    ),
    "notebook_04_status": (
        notebook_04_verification.get(
            "verification_status"
        )
    ),
    "settlement_rows": int(
        len(settlement)
    ),
    "settlement_sha256": sha256_file(
        SETTLEMENT_PATH
    ),
    "shortlist_rows": int(
        len(commercial_shortlist)
    ),
    "commercial_contexts": int(
        commercial_summary[
            [
                "panel",
                "price_area",
                "scenario_stream",
                "nomination_method",
            ]
        ]
        .drop_duplicates()
        .shape[0]
    ),
}

print(
    json.dumps(
        input_manifest,
        indent=2,
    )
)


In [ ]:
# ============================================================
# 4. FOLD LINEAGE AND EVIDENCE AUDIT
# ============================================================

observed_fold_lineage = (
    settlement.groupby(
        [
            "panel",
            "fold_id",
        ],
        as_index=False,
    )
    .agg(
        first_timestamp_utc=(
            "timestamp_utc",
            "min",
        ),
        last_timestamp_utc=(
            "timestamp_utc",
            "max",
        ),
        delivery_days=(
            "delivery_date",
            "nunique",
        ),
        hours=(
            "timestamp_utc",
            "nunique",
        ),
    )
)


if (
    NOTEBOOK_03_CALIBRATION_MANIFEST_PATH
    .exists()
):
    calibration_manifest = pd.read_csv(
        NOTEBOOK_03_CALIBRATION_MANIFEST_PATH
    )

    exact_calibration_lineage = (
        calibration_manifest.loc[
            calibration_manifest[
                "panel"
            ].eq(
                "exact"
            )
        ]
        .copy()
    )
else:
    calibration_manifest = pd.DataFrame()
    exact_calibration_lineage = (
        pd.DataFrame()
    )


panel_evidence = (
    settlement.groupby(
        "panel",
        as_index=False,
    )
    .agg(
        commercial_folds=(
            "fold_id",
            "nunique",
        ),
        commercial_days=(
            "delivery_date",
            "nunique",
        ),
        commercial_hours=(
            "timestamp_utc",
            "nunique",
        ),
    )
)


panel_evidence[
    "evidence_tier"
] = np.select(
    [
        (
            panel_evidence[
                "commercial_folds"
            ]
            >= MINIMUM_PRODUCTION_FOLDS
        )
        & (
            panel_evidence[
                "commercial_days"
            ]
            >= MINIMUM_PRODUCTION_DAYS
        ),
        (
            panel_evidence[
                "commercial_folds"
            ]
            >= 1
        ),
    ],
    [
        "multi_fold_commercial_evidence",
        "single_fold_shadow_evidence",
    ],
    default="insufficient",
)


panel_evidence[
    "interpretation"
] = np.where(
    panel_evidence[
        "panel"
    ].eq(
        "exact"
    ),
    (
        "Notebook 02 exact fold 1 was used "
        "as Notebook 03 calibration history; "
        "exact fold 2 is the only leakage-safe "
        "scenario and commercial evaluation fold."
    ),
    (
        "D2 contains multiple sequential "
        "probabilistic and commercial folds."
    ),
)


display(
    observed_fold_lineage
)

display(
    panel_evidence
)

if not exact_calibration_lineage.empty:
    display(
        exact_calibration_lineage
    )


## Final operating-mode policy map

The policy assignments are deliberately conservative:

### Conservative

The deterministic point nomination is the production benchmark everywhere.

### Balanced

- D2 DK1: raw online risk-adjusted quantile;
- D2 DK2: point nomination;
- exact DK1: point nomination;
- exact DK2: raw q25, but shadow-only because exact has one evaluation fold.

### Profit-seeking

- D2 DK1: raw online-profit quantile;
- D2 DK2: gated q25;
- exact DK1: raw q25;
- exact DK2: raw online-profit quantile.

The exact-track balanced and profit-seeking assignments are research/shadow
policies until another independent exact probabilistic fold is available.


In [ ]:
# ============================================================
# 5. OPERATING-MODE POLICY REGISTRY
# ============================================================

policy_registry = pd.DataFrame(
    [
        # Conservative benchmark.
        {
            "panel": "d2",
            "price_area": "DK1",
            "operating_mode": "conservative",
            "scenario_stream": "raw",
            "nomination_method": "point_centre",
            "policy_role": "production_benchmark",
        },
        {
            "panel": "d2",
            "price_area": "DK2",
            "operating_mode": "conservative",
            "scenario_stream": "raw",
            "nomination_method": "point_centre",
            "policy_role": "production_benchmark",
        },
        {
            "panel": "exact",
            "price_area": "DK1",
            "operating_mode": "conservative",
            "scenario_stream": "raw",
            "nomination_method": "point_centre",
            "policy_role": "shadow_benchmark",
        },
        {
            "panel": "exact",
            "price_area": "DK2",
            "operating_mode": "conservative",
            "scenario_stream": "raw",
            "nomination_method": "point_centre",
            "policy_role": "shadow_benchmark",
        },

        # Balanced mode.
        {
            "panel": "d2",
            "price_area": "DK1",
            "operating_mode": "balanced",
            "scenario_stream": "raw",
            "nomination_method": (
                "online_risk_adjusted_quantile"
            ),
            "policy_role": "controlled_pilot_candidate",
        },
        {
            "panel": "d2",
            "price_area": "DK2",
            "operating_mode": "balanced",
            "scenario_stream": "raw",
            "nomination_method": "point_centre",
            "policy_role": "production_benchmark",
        },
        {
            "panel": "exact",
            "price_area": "DK1",
            "operating_mode": "balanced",
            "scenario_stream": "raw",
            "nomination_method": "point_centre",
            "policy_role": "shadow_benchmark",
        },
        {
            "panel": "exact",
            "price_area": "DK2",
            "operating_mode": "balanced",
            "scenario_stream": "raw",
            "nomination_method": "scenario_q25",
            "policy_role": "shadow_candidate",
        },

        # Profit-seeking mode.
        {
            "panel": "d2",
            "price_area": "DK1",
            "operating_mode": "profit_seeking",
            "scenario_stream": "raw",
            "nomination_method": (
                "online_profit_quantile"
            ),
            "policy_role": "limited_pilot_candidate",
        },
        {
            "panel": "d2",
            "price_area": "DK2",
            "operating_mode": "profit_seeking",
            "scenario_stream": "gated",
            "nomination_method": "scenario_q25",
            "policy_role": "challenger",
        },
        {
            "panel": "exact",
            "price_area": "DK1",
            "operating_mode": "profit_seeking",
            "scenario_stream": "raw",
            "nomination_method": "scenario_q25",
            "policy_role": "shadow_candidate",
        },
        {
            "panel": "exact",
            "price_area": "DK2",
            "operating_mode": "profit_seeking",
            "scenario_stream": "raw",
            "nomination_method": (
                "online_profit_quantile"
            ),
            "policy_role": "shadow_candidate",
        },
    ]
)


registry_key = [
    "panel",
    "price_area",
    "operating_mode",
]


if policy_registry.duplicated(
    registry_key
).any():
    raise RuntimeError(
        "Duplicate operating-mode policy keys."
    )


expected_registry_keys = {
    (
        panel_name,
        price_area,
        operating_mode,
    )
    for panel_name in PANELS
    for price_area in PRICE_AREAS
    for operating_mode in OPERATING_MODES
}


observed_registry_keys = set(
    policy_registry[
        registry_key
    ].itertuples(
        index=False,
        name=None,
    )
)


if (
    observed_registry_keys
    != expected_registry_keys
):
    raise RuntimeError(
        "Policy registry does not cover every "
        "panel-area-mode combination."
    )


available_policy_keys = set(
    settlement[
        [
            "panel",
            "price_area",
            "scenario_stream",
            "nomination_method",
        ]
    ]
    .drop_duplicates()
    .itertuples(
        index=False,
        name=None,
    )
)


missing_policy_keys = [
    tuple(
        row
    )
    for row in policy_registry[
        [
            "panel",
            "price_area",
            "scenario_stream",
            "nomination_method",
        ]
    ].itertuples(
        index=False,
        name=None,
    )
    if tuple(row)
    not in available_policy_keys
]


if missing_policy_keys:
    raise RuntimeError(
        "Policy registry contains unavailable "
        f"settlement policies: {missing_policy_keys}"
    )


display(
    policy_registry.sort_values(
        [
            "panel",
            "operating_mode",
            "price_area",
        ]
    )
)


In [ ]:
# ============================================================
# 6. BUILD INTEGRATED OPERATING-MODE SETTLEMENTS
# ============================================================

mode_parts = []


for policy in policy_registry.itertuples(
    index=False
):
    selected = settlement.loc[
        settlement[
            "panel"
        ].eq(
            policy.panel
        )
        & settlement[
            "price_area"
        ].eq(
            policy.price_area
        )
        & settlement[
            "scenario_stream"
        ].eq(
            policy.scenario_stream
        )
        & settlement[
            "nomination_method"
        ].eq(
            policy.nomination_method
        )
    ].copy()

    if selected.empty:
        raise RuntimeError(
            "No settlement rows for "
            f"{policy.panel}/"
            f"{policy.price_area}/"
            f"{policy.operating_mode}/"
            f"{policy.scenario_stream}/"
            f"{policy.nomination_method}."
        )

    selected[
        "operating_mode"
    ] = policy.operating_mode

    selected[
        "policy_role"
    ] = policy.policy_role

    mode_parts.append(
        selected
    )


mode_settlement = pd.concat(
    mode_parts,
    ignore_index=True,
)


mode_key = [
    "panel",
    "fold_id",
    "timestamp_utc",
    "price_area",
    "operating_mode",
]


if mode_settlement.duplicated(
    mode_key
).any():
    raise RuntimeError(
        "Duplicate integrated operating-mode keys."
    )


mode_support = (
    mode_settlement.groupby(
        [
            "panel",
            "operating_mode",
            "price_area",
        ]
    )[
        "timestamp_utc"
    ]
    .nunique()
    .unstack(
        "price_area"
    )
)


if not (
    mode_support["DK1"]
    == mode_support["DK2"]
).all():
    raise RuntimeError(
        "DK1 and DK2 do not share common "
        "support inside an operating mode."
    )


mode_settlement.to_parquet(
    TABLE_DIR
    / "notebook_05_mode_settlement_hourly.parquet",
    index=False,
)


print(
    {
        "mode_settlement_rows": int(
            len(mode_settlement)
        ),
        "mode_contexts": int(
            mode_settlement[
                [
                    "panel",
                    "operating_mode",
                    "price_area",
                ]
            ]
            .drop_duplicates()
            .shape[0]
        ),
    }
)


In [ ]:
# ============================================================
# 7. OPERATING-MODE COMMERCIAL METRICS
# ============================================================

def downside_cvar(
    values: np.ndarray,
    probability: float,
) -> float:
    array = np.asarray(
        values,
        dtype=float,
    )

    array = array[
        np.isfinite(array)
    ]

    if len(array) == 0:
        return np.nan

    threshold = np.quantile(
        array,
        probability,
    )

    tail = array[
        array <= threshold
    ]

    return float(
        tail.mean()
        if len(tail)
        else threshold
    )


mode_hourly_portfolio = (
    mode_settlement.groupby(
        [
            "panel",
            "fold_id",
            "delivery_date",
            "timestamp_utc",
            "operating_mode",
        ],
        as_index=False,
    )
    .agg(
        actual_mwh=(
            "actual_mwh",
            "sum",
        ),
        nominated_mwh=(
            "nominated_mwh",
            "sum",
        ),
        signed_imbalance_mwh=(
            "imbalance_mwh",
            "sum",
        ),
        area_absolute_imbalance_mwh=(
            "absolute_imbalance_mwh",
            "sum",
        ),
        gross_revenue_eur=(
            "gross_revenue_eur",
            "sum",
        ),
        perfect_revenue_eur=(
            "perfect_nomination_revenue_eur",
            "sum",
        ),
        incremental_eur=(
            "net_incremental_revenue_vs_perfect_eur",
            "sum",
        ),
    )
)


mode_hourly_portfolio[
    "portfolio_absolute_imbalance_mwh"
] = mode_hourly_portfolio[
    "signed_imbalance_mwh"
].abs()


mode_daily_portfolio = (
    mode_hourly_portfolio.groupby(
        [
            "panel",
            "fold_id",
            "delivery_date",
            "operating_mode",
        ],
        as_index=False,
    )
    .agg(
        daily_actual_mwh=(
            "actual_mwh",
            "sum",
        ),
        daily_area_absolute_imbalance_mwh=(
            "area_absolute_imbalance_mwh",
            "sum",
        ),
        daily_portfolio_absolute_imbalance_mwh=(
            "portfolio_absolute_imbalance_mwh",
            "sum",
        ),
        daily_incremental_eur=(
            "incremental_eur",
            "sum",
        ),
    )
)


mode_summary_rows = []


for keys, group in mode_hourly_portfolio.groupby(
    [
        "panel",
        "operating_mode",
    ],
    sort=True,
):
    (
        panel_name,
        operating_mode,
    ) = keys

    daily = mode_daily_portfolio.loc[
        mode_daily_portfolio[
            "panel"
        ].eq(
            panel_name
        )
        & mode_daily_portfolio[
            "operating_mode"
        ].eq(
            operating_mode
        )
    ]

    total_actual = float(
        group[
            "actual_mwh"
        ].sum()
    )

    total_incremental = float(
        group[
            "incremental_eur"
        ].sum()
    )

    total_gross = float(
        group[
            "gross_revenue_eur"
        ].sum()
    )

    total_perfect = float(
        group[
            "perfect_revenue_eur"
        ].sum()
    )

    mode_summary_rows.append(
        {
            "panel": panel_name,
            "operating_mode": (
                operating_mode
            ),
            "folds": int(
                group[
                    "fold_id"
                ].nunique()
            ),
            "days": int(
                group[
                    "delivery_date"
                ].nunique()
            ),
            "hours": int(
                group[
                    "timestamp_utc"
                ].nunique()
            ),
            "total_actual_mwh": (
                total_actual
            ),
            "total_incremental_eur": (
                total_incremental
            ),
            "incremental_eur_per_actual_mwh": (
                total_incremental
                / total_actual
                if total_actual > 0
                else np.nan
            ),
            "gross_revenue_capture_pct": (
                100.0
                * total_gross
                / total_perfect
                if abs(
                    total_perfect
                )
                > 1e-12
                else np.nan
            ),
            "total_area_absolute_imbalance_mwh": float(
                group[
                    "area_absolute_imbalance_mwh"
                ].sum()
            ),
            "total_portfolio_absolute_imbalance_mwh": float(
                group[
                    "portfolio_absolute_imbalance_mwh"
                ].sum()
            ),
            "mean_daily_incremental_eur": float(
                daily[
                    "daily_incremental_eur"
                ].mean()
            ),
            "daily_downside_cvar_5pct_eur": (
                downside_cvar(
                    daily[
                        "daily_incremental_eur"
                    ].to_numpy(
                        dtype=float
                    ),
                    0.05,
                )
            ),
            "worst_day_incremental_eur": float(
                daily[
                    "daily_incremental_eur"
                ].min()
            ),
        }
    )


mode_summary = pd.DataFrame(
    mode_summary_rows
)


conservative_reference = (
    mode_summary.loc[
        mode_summary[
            "operating_mode"
        ].eq(
            "conservative"
        ),
        [
            "panel",
            "total_incremental_eur",
            "incremental_eur_per_actual_mwh",
            "total_area_absolute_imbalance_mwh",
            "total_portfolio_absolute_imbalance_mwh",
        ],
    ]
    .rename(
        columns={
            "total_incremental_eur": (
                "conservative_total_incremental_eur"
            ),
            "incremental_eur_per_actual_mwh": (
                "conservative_incremental_eur_per_actual_mwh"
            ),
            "total_area_absolute_imbalance_mwh": (
                "conservative_area_absolute_imbalance_mwh"
            ),
            "total_portfolio_absolute_imbalance_mwh": (
                "conservative_portfolio_absolute_imbalance_mwh"
            ),
        }
    )
)


mode_summary = mode_summary.merge(
    conservative_reference,
    on="panel",
    how="left",
    validate="many_to_one",
)


mode_summary[
    "commercial_delta_vs_conservative_eur"
] = (
    mode_summary[
        "total_incremental_eur"
    ]
    - mode_summary[
        "conservative_total_incremental_eur"
    ]
)


mode_summary[
    "commercial_delta_vs_conservative_eur_per_mwh"
] = (
    mode_summary[
        "incremental_eur_per_actual_mwh"
    ]
    - mode_summary[
        "conservative_incremental_eur_per_actual_mwh"
    ]
)


mode_summary[
    "area_absolute_imbalance_change_pct"
] = (
    100.0
    * (
        mode_summary[
            "total_area_absolute_imbalance_mwh"
        ]
        / mode_summary[
            "conservative_area_absolute_imbalance_mwh"
        ]
        - 1.0
    )
)


mode_summary[
    "portfolio_absolute_imbalance_change_pct"
] = (
    100.0
    * (
        mode_summary[
            "total_portfolio_absolute_imbalance_mwh"
        ]
        / mode_summary[
            "conservative_portfolio_absolute_imbalance_mwh"
        ]
        - 1.0
    )
)


mode_summary = (
    mode_summary.sort_values(
        [
            "panel",
            "operating_mode",
        ]
    )
    .reset_index(
        drop=True
    )
)


display(
    mode_summary[
        [
            "panel",
            "operating_mode",
            "folds",
            "days",
            "total_incremental_eur",
            "incremental_eur_per_actual_mwh",
            "commercial_delta_vs_conservative_eur",
            "commercial_delta_vs_conservative_eur_per_mwh",
            "total_area_absolute_imbalance_mwh",
            "area_absolute_imbalance_change_pct",
            "daily_downside_cvar_5pct_eur",
            "worst_day_incremental_eur",
        ]
    ]
)


In [ ]:
# ============================================================
# 8. POLICY-LEVEL PROMOTION GATES
# ============================================================

point_baselines = (
    commercial_summary.loc[
        commercial_summary[
            "nomination_method"
        ].eq(
            "point_centre"
        ),
        [
            "panel",
            "price_area",
            "scenario_stream",
            "total_absolute_imbalance_mwh",
        ],
    ]
    .rename(
        columns={
            "total_absolute_imbalance_mwh": (
                "point_absolute_imbalance_mwh"
            )
        }
    )
)


policy_evidence = (
    policy_registry.merge(
        comparison_vs_point,
        on=[
            "panel",
            "price_area",
            "scenario_stream",
            "nomination_method",
        ],
        how="left",
        validate="many_to_one",
    )
    .merge(
        inference_vs_point,
        on=[
            "panel",
            "price_area",
            "scenario_stream",
            "nomination_method",
        ],
        how="left",
        validate="many_to_one",
        suffixes=(
            "",
            "_inference",
        ),
    )
    .merge(
        point_baselines,
        on=[
            "panel",
            "price_area",
            "scenario_stream",
        ],
        how="left",
        validate="many_to_one",
    )
)


policy_evidence[
    "is_reference_policy"
] = policy_evidence[
    "nomination_method"
].eq(
    "point_centre"
)


policy_evidence[
    "imbalance_increase_pct_vs_point"
] = (
    100.0
    * policy_evidence[
        "total_absolute_imbalance_delta_vs_point_mwh"
    ]
    / policy_evidence[
        "point_absolute_imbalance_mwh"
    ]
)


policy_evidence[
    "positive_commercial_delta"
] = (
    policy_evidence[
        "total_commercial_delta_vs_point_eur"
    ]
    > 0
)


policy_evidence[
    "minimum_fold_gate"
] = (
    policy_evidence[
        "folds_evaluated"
    ]
    >= MINIMUM_PRODUCTION_FOLDS
)


policy_evidence[
    "minimum_day_gate"
] = (
    policy_evidence[
        "days"
    ]
    >= MINIMUM_PRODUCTION_DAYS
)


policy_evidence[
    "all_folds_won_gate"
] = (
    policy_evidence[
        "folds_won_vs_point"
    ]
    == policy_evidence[
        "folds_evaluated"
    ]
)


policy_evidence[
    "pilot_probability_gate"
] = (
    policy_evidence[
        "bootstrap_probability_outperformance"
    ]
    >= MINIMUM_PILOT_BOOTSTRAP_PROBABILITY
)


policy_evidence[
    "production_probability_gate"
] = (
    policy_evidence[
        "bootstrap_probability_outperformance"
    ]
    >= MINIMUM_PRODUCTION_BOOTSTRAP_PROBABILITY
)


policy_evidence[
    "positive_confidence_bound_gate"
] = (
    policy_evidence[
        "bootstrap_ci_lower_mean_daily_eur"
    ]
    > 0
)


policy_evidence[
    "multiple_testing_gate"
] = (
    policy_evidence[
        "holm_bootstrap_p_value"
    ]
    < 0.05
)


policy_evidence[
    "pilot_imbalance_gate"
] = (
    policy_evidence[
        "imbalance_increase_pct_vs_point"
    ]
    <= MAXIMUM_PILOT_IMBALANCE_INCREASE_PCT
)


policy_evidence[
    "production_imbalance_gate"
] = (
    policy_evidence[
        "imbalance_increase_pct_vs_point"
    ]
    <= MAXIMUM_PRODUCTION_IMBALANCE_INCREASE_PCT
)


def assign_evidence_status(
    row: pd.Series,
) -> str:
    if bool(
        row["is_reference_policy"]
    ):
        return "reference"

    if (
        row["panel"] == "exact"
        or not bool(
            row["minimum_fold_gate"]
        )
    ):
        return "shadow_only"

    production_gates = [
        row[
            "positive_commercial_delta"
        ],
        row[
            "minimum_day_gate"
        ],
        row[
            "all_folds_won_gate"
        ],
        row[
            "production_probability_gate"
        ],
        row[
            "positive_confidence_bound_gate"
        ],
        row[
            "multiple_testing_gate"
        ],
        row[
            "production_imbalance_gate"
        ],
    ]

    if all(
        bool(value)
        for value in production_gates
    ):
        return "production_eligible"

    pilot_gates = [
        row[
            "positive_commercial_delta"
        ],
        row[
            "minimum_day_gate"
        ],
        row[
            "all_folds_won_gate"
        ],
        row[
            "pilot_probability_gate"
        ],
        row[
            "pilot_imbalance_gate"
        ],
    ]

    if all(
        bool(value)
        for value in pilot_gates
    ):
        return "pilot_eligible"

    if bool(
        row[
            "positive_commercial_delta"
        ]
    ):
        return "challenger_only"

    return "reject"


policy_evidence[
    "evidence_status"
] = policy_evidence.apply(
    assign_evidence_status,
    axis=1,
)


display(
    policy_evidence[
        [
            "panel",
            "price_area",
            "operating_mode",
            "scenario_stream",
            "nomination_method",
            "total_commercial_delta_vs_point_eur",
            "folds_won_vs_point",
            "folds_evaluated",
            "days",
            "bootstrap_probability_outperformance",
            "bootstrap_ci_lower_mean_daily_eur",
            "bootstrap_ci_upper_mean_daily_eur",
            "holm_bootstrap_p_value",
            "imbalance_increase_pct_vs_point",
            "evidence_status",
        ]
    ].sort_values(
        [
            "panel",
            "operating_mode",
            "price_area",
        ]
    )
)


In [ ]:
# ============================================================
# 9. APPROXIMATE ADDITIONAL EVIDENCE REQUIRED
# ============================================================

evidence_horizon = policy_evidence.loc[
    ~policy_evidence[
        "is_reference_policy"
    ]
].copy()


absolute_t = evidence_horizon[
    "hac_t_statistic"
].abs()


evidence_horizon[
    "approx_total_days_for_unadjusted_95pct"
] = np.where(
    absolute_t > 1e-12,
    np.ceil(
        evidence_horizon[
            "days"
        ]
        * (
            TWO_SIDED_95_CRITICAL_VALUE
            / absolute_t
        ) ** 2
    ),
    np.inf,
)


evidence_horizon[
    "approx_total_days_for_conservative_threshold"
] = np.where(
    absolute_t > 1e-12,
    np.ceil(
        evidence_horizon[
            "days"
        ]
        * (
            CONSERVATIVE_MULTIPLE_TEST_CRITICAL_VALUE
            / absolute_t
        ) ** 2
    ),
    np.inf,
)


evidence_horizon[
    "approx_additional_days_unadjusted"
] = np.maximum(
    evidence_horizon[
        "approx_total_days_for_unadjusted_95pct"
    ]
    - evidence_horizon[
        "days"
    ],
    0,
)


evidence_horizon[
    "approx_additional_days_conservative"
] = np.maximum(
    evidence_horizon[
        "approx_total_days_for_conservative_threshold"
    ]
    - evidence_horizon[
        "days"
    ],
    0,
)


evidence_horizon[
    "evidence_horizon_caveat"
] = (
    "Indicative only: assumes the observed "
    "effect size, dependence and variance "
    "remain stable; regime changes can "
    "invalidate the extrapolation."
)


display(
    evidence_horizon[
        [
            "panel",
            "price_area",
            "operating_mode",
            "nomination_method",
            "days",
            "hac_t_statistic",
            "approx_total_days_for_unadjusted_95pct",
            "approx_additional_days_unadjusted",
            "approx_total_days_for_conservative_threshold",
            "approx_additional_days_conservative",
            "evidence_status",
        ]
    ].sort_values(
        [
            "panel",
            "operating_mode",
            "price_area",
        ]
    )
)


In [ ]:
# ============================================================
# 10. FEE AND PRICE-SPREAD STRESS TESTS BY OPERATING MODE
# ============================================================

mode_sensitivity = (
    commercial_sensitivity.merge(
        policy_registry,
        on=[
            "panel",
            "price_area",
            "scenario_stream",
            "nomination_method",
        ],
        how="inner",
        validate="many_to_many",
    )
)


mode_sensitivity_summary = (
    mode_sensitivity.groupby(
        [
            "panel",
            "operating_mode",
            "additional_fee_eur_per_mwh",
            "spread_multiplier",
        ],
        as_index=False,
    )
    .agg(
        total_stressed_incremental_eur=(
            "total_stressed_incremental_eur",
            "sum",
        ),
        total_absolute_imbalance_mwh=(
            "total_absolute_imbalance_mwh",
            "sum",
        ),
    )
)


stress_conservative = (
    mode_sensitivity_summary.loc[
        mode_sensitivity_summary[
            "operating_mode"
        ].eq(
            "conservative"
        ),
        [
            "panel",
            "additional_fee_eur_per_mwh",
            "spread_multiplier",
            "total_stressed_incremental_eur",
        ],
    ]
    .rename(
        columns={
            "total_stressed_incremental_eur": (
                "conservative_stressed_incremental_eur"
            )
        }
    )
)


mode_sensitivity_summary = (
    mode_sensitivity_summary.merge(
        stress_conservative,
        on=[
            "panel",
            "additional_fee_eur_per_mwh",
            "spread_multiplier",
        ],
        how="left",
        validate="many_to_one",
    )
)


mode_sensitivity_summary[
    "stressed_commercial_delta_vs_conservative_eur"
] = (
    mode_sensitivity_summary[
        "total_stressed_incremental_eur"
    ]
    - mode_sensitivity_summary[
        "conservative_stressed_incremental_eur"
    ]
)


stress_resilience = (
    mode_sensitivity_summary.groupby(
        [
            "panel",
            "operating_mode",
        ],
        as_index=False,
    )
    .agg(
        stress_cases=(
            "stressed_commercial_delta_vs_conservative_eur",
            "size",
        ),
        positive_stress_cases=(
            "stressed_commercial_delta_vs_conservative_eur",
            lambda values: int(
                (
                    values
                    > 0
                ).sum()
            ),
        ),
        minimum_stressed_delta_vs_conservative_eur=(
            "stressed_commercial_delta_vs_conservative_eur",
            "min",
        ),
        median_stressed_delta_vs_conservative_eur=(
            "stressed_commercial_delta_vs_conservative_eur",
            "median",
        ),
        maximum_stressed_delta_vs_conservative_eur=(
            "stressed_commercial_delta_vs_conservative_eur",
            "max",
        ),
    )
)


stress_resilience[
    "positive_stress_case_share"
] = (
    stress_resilience[
        "positive_stress_cases"
    ]
    / stress_resilience[
        "stress_cases"
    ]
)


display(
    stress_resilience
)


In [ ]:
# ============================================================
# 11. FINAL DEPLOYMENT RECOMMENDATION
# ============================================================

deployment_recommendation = pd.DataFrame(
    [
        {
            "panel": "d2",
            "operating_mode": "conservative",
            "deployment_status": (
                "production_baseline"
            ),
            "recommended_use": (
                "Default live benchmark and automatic fallback."
            ),
            "reason": (
                "Information-safe deterministic nomination "
                "with the lowest governance burden."
            ),
        },
        {
            "panel": "d2",
            "operating_mode": "balanced",
            "deployment_status": (
                "controlled_pilot"
            ),
            "recommended_use": (
                "Shadow first, then limited-volume live pilot "
                "for DK1 while DK2 remains on the point forecast."
            ),
            "reason": (
                "D2 DK1 raw online risk-adjusted policy "
                "has positive multi-fold commercial evidence "
                "and lower exposure than the pure-profit policy, "
                "but no 5% statistical proof."
            ),
        },
        {
            "panel": "d2",
            "operating_mode": "profit_seeking",
            "deployment_status": (
                "limited_pilot_only"
            ),
            "recommended_use": (
                "Capped-volume trader overlay with strict "
                "daily loss and imbalance limits."
            ),
            "reason": (
                "Highest observed D2 commercial value, "
                "but materially higher imbalance exposure "
                "and confidence intervals crossing zero."
            ),
        },
        {
            "panel": "exact",
            "operating_mode": "conservative",
            "deployment_status": (
                "shadow_baseline"
            ),
            "recommended_use": (
                "Run as a forecast-horizon benchmark; "
                "do not claim production superiority."
            ),
            "reason": (
                "Only one leakage-safe probabilistic and "
                "commercial evaluation fold."
            ),
        },
        {
            "panel": "exact",
            "operating_mode": "balanced",
            "deployment_status": (
                "shadow_only"
            ),
            "recommended_use": (
                "Record hypothetical nominations and P&L "
                "until a second independent exact fold exists."
            ),
            "reason": (
                "Commercially interesting DK2 q25 result, "
                "but only 40 days and one fold."
            ),
        },
        {
            "panel": "exact",
            "operating_mode": "profit_seeking",
            "deployment_status": (
                "shadow_only"
            ),
            "recommended_use": (
                "Research and trader-review overlay only."
            ),
            "reason": (
                "Large point estimate in exact DK2, "
                "but insufficient independent evidence."
            ),
        },
    ]
)


deployment_recommendation = (
    deployment_recommendation.merge(
        mode_summary[
            [
                "panel",
                "operating_mode",
                "commercial_delta_vs_conservative_eur",
                "commercial_delta_vs_conservative_eur_per_mwh",
                "area_absolute_imbalance_change_pct",
                "daily_downside_cvar_5pct_eur",
                "worst_day_incremental_eur",
            ]
        ],
        on=[
            "panel",
            "operating_mode",
        ],
        how="left",
        validate="one_to_one",
    )
    .merge(
        stress_resilience[
            [
                "panel",
                "operating_mode",
                "positive_stress_case_share",
                "minimum_stressed_delta_vs_conservative_eur",
                "median_stressed_delta_vs_conservative_eur",
            ]
        ],
        on=[
            "panel",
            "operating_mode",
        ],
        how="left",
        validate="one_to_one",
    )
)


display(
    deployment_recommendation[
        [
            "panel",
            "operating_mode",
            "deployment_status",
            "commercial_delta_vs_conservative_eur",
            "commercial_delta_vs_conservative_eur_per_mwh",
            "area_absolute_imbalance_change_pct",
            "positive_stress_case_share",
            "recommended_use",
        ]
    ]
)


## Operational fallback hierarchy

Every non-reference policy must fall back to the deterministic point nomination
when any critical input or monitoring condition fails.

Fallback order:

1. selected policy in the active mode;
2. same-panel point nomination;
3. most recent valid point nomination with an explicit stale-data flag;
4. manual trader intervention.

No probabilistic tail nomination should be produced when scenario quantiles,
market data, or model-origin timestamps fail validation.


In [ ]:
# ============================================================
# 12. MONITORING AND FALLBACK CONFIGURATION
# ============================================================

monitoring_rules = pd.DataFrame(
    [
        {
            "rule_id": (
                "market_data_completeness"
            ),
            "metric": (
                "rolling_7d_complete_market_hour_share"
            ),
            "comparison": ">=",
            "threshold": (
                MINIMUM_MARKET_DATA_COMPLETENESS
            ),
            "action_on_failure": (
                "fallback_to_point_and_alert"
            ),
            "severity": "critical",
        },
        {
            "rule_id": (
                "commercial_delta"
            ),
            "metric": (
                "rolling_28d_commercial_delta_"
                "vs_point_eur_per_mwh"
            ),
            "comparison": ">=",
            "threshold": (
                MINIMUM_ROLLING_COMMERCIAL_DELTA_EUR_PER_MWH
            ),
            "action_on_failure": (
                "fallback_to_point_and_review"
            ),
            "severity": "critical",
        },
        {
            "rule_id": (
                "imbalance_increase"
            ),
            "metric": (
                "rolling_28d_absolute_imbalance_"
                "increase_vs_point_pct"
            ),
            "comparison": "<=",
            "threshold": (
                MAXIMUM_ROLLING_IMBALANCE_INCREASE_PCT
            ),
            "action_on_failure": (
                "reduce_risk_mode_or_fallback"
            ),
            "severity": "high",
        },
        {
            "rule_id": (
                "selected_quantile_lower_bound"
            ),
            "metric": (
                "selected_nomination_quantile"
            ),
            "comparison": ">=",
            "threshold": (
                MINIMUM_SELECTED_QUANTILE
            ),
            "action_on_failure": (
                "clip_or_fallback_to_point"
            ),
            "severity": "high",
        },
        {
            "rule_id": (
                "selected_quantile_upper_bound"
            ),
            "metric": (
                "selected_nomination_quantile"
            ),
            "comparison": "<=",
            "threshold": (
                MAXIMUM_SELECTED_QUANTILE
            ),
            "action_on_failure": (
                "clip_or_fallback_to_point"
            ),
            "severity": "high",
        },
        {
            "rule_id": (
                "scenario_availability"
            ),
            "metric": (
                "required_scenario_quantiles_available"
            ),
            "comparison": "==",
            "threshold": 1.0,
            "action_on_failure": (
                "fallback_to_point_and_alert"
            ),
            "severity": "critical",
        },
        {
            "rule_id": (
                "forecast_origin_integrity"
            ),
            "metric": (
                "forecast_origin_before_gate_closure"
            ),
            "comparison": "==",
            "threshold": 1.0,
            "action_on_failure": (
                "block_nomination_and_alert"
            ),
            "severity": "critical",
        },
    ]
)


fallback_hierarchy = pd.DataFrame(
    [
        {
            "priority": 1,
            "policy": (
                "active_operating_mode_policy"
            ),
            "condition": (
                "all critical checks pass"
            ),
        },
        {
            "priority": 2,
            "policy": (
                "same_panel_point_centre"
            ),
            "condition": (
                "active policy unavailable or "
                "monitoring gate fails"
            ),
        },
        {
            "priority": 3,
            "policy": (
                "last_valid_point_centre"
            ),
            "condition": (
                "current point forecast unavailable; "
                "must carry stale-data flag"
            ),
        },
        {
            "priority": 4,
            "policy": (
                "manual_trader_intervention"
            ),
            "condition": (
                "no validated automated nomination exists"
            ),
        },
    ]
)


display(
    monitoring_rules
)

display(
    fallback_hierarchy
)


In [ ]:
# ============================================================
# 13. SHADOW-TRADING AND PROMOTION PLAN
# ============================================================

promotion_plan = pd.DataFrame(
    [
        {
            "stage": 0,
            "name": "historical_validation",
            "minimum_duration_days": 0,
            "capital_or_volume_share": 0.0,
            "requirements": (
                "Notebooks 01–05 complete; "
                "all audit gates pass."
            ),
            "promotion_decision": (
                "Proceed to live shadow operation."
            ),
        },
        {
            "stage": 1,
            "name": "live_shadow",
            "minimum_duration_days": 90,
            "capital_or_volume_share": 0.0,
            "requirements": (
                "Daily live forecasts, nominations, "
                "counterfactual settlement and incident logs."
            ),
            "promotion_decision": (
                "Require positive rolling commercial delta, "
                "stable data quality and no severe control failure."
            ),
        },
        {
            "stage": 2,
            "name": "limited_volume_pilot",
            "minimum_duration_days": 90,
            "capital_or_volume_share": 0.10,
            "requirements": (
                "At least two independent live regimes or folds; "
                "pilot policy beats point benchmark on average; "
                "risk limits pass."
            ),
            "promotion_decision": (
                "Increase volume only after governance approval."
            ),
        },
        {
            "stage": 3,
            "name": "scaled_pilot",
            "minimum_duration_days": 180,
            "capital_or_volume_share": 0.25,
            "requirements": (
                "Positive lower confidence bound or equivalent "
                "pre-registered evidence; fee stress resilient; "
                "no unresolved operational incident."
            ),
            "promotion_decision": (
                "Eligible for production review."
            ),
        },
        {
            "stage": 4,
            "name": "production",
            "minimum_duration_days": 0,
            "capital_or_volume_share": 1.0,
            "requirements": (
                "Formal model-risk approval, trader sign-off, "
                "real fee model, intraday process and rollback plan."
            ),
            "promotion_decision": (
                "Continuous monitoring with automatic fallback."
            ),
        },
    ]
)


display(
    promotion_plan
)


In [ ]:
# ============================================================
# 14. DEPLOYMENT CONFIGURATION AND LOOKUP FUNCTION
# ============================================================

deployment_policy_registry = (
    policy_registry.merge(
        policy_evidence[
            [
                "panel",
                "price_area",
                "operating_mode",
                "scenario_stream",
                "nomination_method",
                "evidence_status",
            ]
        ],
        on=[
            "panel",
            "price_area",
            "operating_mode",
            "scenario_stream",
            "nomination_method",
        ],
        how="left",
        validate="one_to_one",
    )
)


deployment_policy_registry[
    "live_enabled"
] = (
    deployment_policy_registry[
        "evidence_status"
    ].isin(
        [
            "reference",
            "production_eligible",
        ]
    )
)


deployment_policy_registry[
    "pilot_enabled"
] = (
    deployment_policy_registry[
        "evidence_status"
    ].isin(
        [
            "pilot_eligible",
            "production_eligible",
        ]
    )
)


# Exact policies remain shadow-only regardless of retrospective rank.
deployment_policy_registry.loc[
    deployment_policy_registry[
        "panel"
    ].eq(
        "exact"
    ),
    [
        "live_enabled",
        "pilot_enabled",
    ],
] = False


def get_operating_policy(
    panel: str,
    price_area: str,
    operating_mode: str,
) -> dict[str, Any]:
    selected = (
        deployment_policy_registry.loc[
            deployment_policy_registry[
                "panel"
            ].eq(
                panel
            )
            & deployment_policy_registry[
                "price_area"
            ].eq(
                price_area
            )
            & deployment_policy_registry[
                "operating_mode"
            ].eq(
                operating_mode
            )
        ]
    )

    if len(selected) != 1:
        raise KeyError(
            "Expected one operating policy for "
            f"{panel}/{price_area}/"
            f"{operating_mode}; found "
            f"{len(selected)}."
        )

    return (
        selected.iloc[0]
        .to_dict()
    )


deployment_config = {
    "system_name": (
        "Nordic Wind Decision System"
    ),
    "version": "1.0",
    "default_live_mode": (
        "conservative"
    ),
    "recommended_d2_pilot_mode": (
        "balanced"
    ),
    "exact_track_live_status": (
        "shadow_only_until_second_"
        "probabilistic_commercial_fold"
    ),
    "policy_registry": (
        deployment_policy_registry
        .sort_values(
            [
                "panel",
                "operating_mode",
                "price_area",
            ]
        )
        .to_dict(
            orient="records"
        )
    ),
    "monitoring_rules": (
        monitoring_rules.to_dict(
            orient="records"
        )
    ),
    "fallback_hierarchy": (
        fallback_hierarchy.to_dict(
            orient="records"
        )
    ),
}


print(
    json.dumps(
        {
            "default_live_mode": (
                deployment_config[
                    "default_live_mode"
                ]
            ),
            "recommended_d2_pilot_mode": (
                deployment_config[
                    "recommended_d2_pilot_mode"
                ]
            ),
            "exact_track_live_status": (
                deployment_config[
                    "exact_track_live_status"
                ]
            ),
            "d2_dk1_balanced": (
                get_operating_policy(
                    "d2",
                    "DK1",
                    "balanced",
                )
            ),
        },
        indent=2,
        default=str,
    )
)


In [ ]:
# ============================================================
# 15. VISUAL DIAGNOSTICS
# ============================================================

def save_current_figure(
    filename: str,
) -> None:
    path = (
        FIGURE_DIR
        / filename
    )

    plt.tight_layout()

    plt.savefig(
        path,
        dpi=160,
        bbox_inches="tight",
    )

    plt.show()
    plt.close()


for panel_name in PANELS:
    selected = (
        mode_summary.loc[
            mode_summary[
                "panel"
            ].eq(
                panel_name
            )
        ]
        .sort_values(
            "commercial_delta_vs_conservative_eur"
        )
    )

    plt.figure(
        figsize=(10, 6)
    )

    plt.barh(
        selected[
            "operating_mode"
        ],
        selected[
            "commercial_delta_vs_conservative_eur"
        ],
    )

    plt.xlabel(
        "Commercial delta versus conservative mode (EUR)"
    )

    plt.ylabel(
        "Operating mode"
    )

    plt.title(
        f"{panel_name.upper()}: "
        "integrated operating-mode value"
    )

    save_current_figure(
        (
            "notebook_05_mode_value_"
            f"{panel_name}.png"
        )
    )


for panel_name in PANELS:
    selected = (
        mode_summary.loc[
            mode_summary[
                "panel"
            ].eq(
                panel_name
            )
        ]
        .sort_values(
            "area_absolute_imbalance_change_pct"
        )
    )

    plt.figure(
        figsize=(10, 6)
    )

    plt.barh(
        selected[
            "operating_mode"
        ],
        selected[
            "area_absolute_imbalance_change_pct"
        ],
    )

    plt.xlabel(
        "Area-level absolute imbalance change "
        "versus conservative mode (%)"
    )

    plt.ylabel(
        "Operating mode"
    )

    plt.title(
        f"{panel_name.upper()}: "
        "commercial value–imbalance trade-off"
    )

    save_current_figure(
        (
            "notebook_05_mode_imbalance_"
            f"{panel_name}.png"
        )
    )


In [ ]:
# ============================================================
# 16. SAVE FINAL ARTIFACTS
# ============================================================

policy_registry.to_csv(
    CONFIG_DIR
    / "operating_mode_policy_registry.csv",
    index=False,
)

deployment_policy_registry.to_csv(
    CONFIG_DIR
    / "deployment_policy_registry.csv",
    index=False,
)

mode_summary.to_csv(
    TABLE_DIR
    / "operating_mode_commercial_summary.csv",
    index=False,
)

mode_daily_portfolio.to_parquet(
    TABLE_DIR
    / "operating_mode_daily_portfolio.parquet",
    index=False,
)

policy_evidence.to_csv(
    TABLE_DIR
    / "policy_promotion_gate_evaluation.csv",
    index=False,
)

evidence_horizon.to_csv(
    TABLE_DIR
    / "approximate_evidence_horizon.csv",
    index=False,
)

mode_sensitivity_summary.to_csv(
    TABLE_DIR
    / "operating_mode_fee_spread_sensitivity.csv",
    index=False,
)

stress_resilience.to_csv(
    TABLE_DIR
    / "operating_mode_stress_resilience.csv",
    index=False,
)

deployment_recommendation.to_csv(
    TABLE_DIR
    / "final_deployment_recommendation.csv",
    index=False,
)

panel_evidence.to_csv(
    AUDIT_DIR
    / "panel_evidence_tiers.csv",
    index=False,
)

observed_fold_lineage.to_csv(
    AUDIT_DIR
    / "observed_commercial_fold_lineage.csv",
    index=False,
)

monitoring_rules.to_csv(
    MONITORING_DIR
    / "monitoring_rules.csv",
    index=False,
)

fallback_hierarchy.to_csv(
    MONITORING_DIR
    / "fallback_hierarchy.csv",
    index=False,
)

promotion_plan.to_csv(
    MONITORING_DIR
    / "shadow_trading_promotion_plan.csv",
    index=False,
)


deployment_config_path = (
    CONFIG_DIR
    / "deployment_config.json"
)


with deployment_config_path.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        deployment_config,
        file,
        indent=2,
        default=str,
    )


print(
    {
        "deployment_config": str(
            deployment_config_path
        ),
        "policy_registry_rows": int(
            len(
                deployment_policy_registry
            )
        ),
        "mode_summary_rows": int(
            len(
                mode_summary
            )
        ),
        "promotion_gate_rows": int(
            len(
                policy_evidence
            )
        ),
    }
)


In [ ]:
# ============================================================
# 17. FINAL VERIFICATION
# ============================================================

blocking_failures = []


if notebook_03_verification.get(
    "verification_status"
) != "PROBABILISTIC_MODELLING_COMPLETE":
    blocking_failures.append(
        "Notebook 03 verification failed."
    )


if notebook_04_verification.get(
    "verification_status"
) != "COMMERCIAL_EVALUATION_COMPLETE":
    blocking_failures.append(
        "Notebook 04 verification failed."
    )


if notebook_04_verification.get(
    "blocking_failures"
):
    blocking_failures.append(
        "Notebook 04 contains blocking failures."
    )


if (
    observed_registry_keys
    != expected_registry_keys
):
    blocking_failures.append(
        "Policy registry coverage is incomplete."
    )


if missing_policy_keys:
    blocking_failures.append(
        "Policy registry contains unavailable policies."
    )


if mode_settlement.empty:
    blocking_failures.append(
        "Integrated mode settlement is empty."
    )


if mode_summary.empty:
    blocking_failures.append(
        "Operating-mode summary is empty."
    )


if policy_evidence.empty:
    blocking_failures.append(
        "Promotion-gate evaluation is empty."
    )


if deployment_recommendation.empty:
    blocking_failures.append(
        "Deployment recommendation is empty."
    )


if not deployment_config_path.exists():
    blocking_failures.append(
        "Deployment configuration was not saved."
    )


production_eligible_non_reference = int(
    (
        policy_evidence[
            "evidence_status"
        ].eq(
            "production_eligible"
        )
        & ~policy_evidence[
            "is_reference_policy"
        ]
    ).sum()
)


pilot_eligible_non_reference = int(
    (
        policy_evidence[
            "evidence_status"
        ].eq(
            "pilot_eligible"
        )
        & ~policy_evidence[
            "is_reference_policy"
        ]
    ).sum()
)


exact_non_reference_live_enabled = int(
    (
        deployment_policy_registry[
            "panel"
        ].eq(
            "exact"
        )
        & ~deployment_policy_registry[
            "nomination_method"
        ].eq(
            "point_centre"
        )
        & deployment_policy_registry[
            "live_enabled"
        ]
    ).sum()
)


if exact_non_reference_live_enabled:
    blocking_failures.append(
        "An exact non-reference policy is incorrectly "
        "enabled for live deployment."
    )


verification_status = (
    "INTEGRATED_DECISION_SYSTEM_COMPLETE"
    if not blocking_failures
    else "BLOCKED"
)


def file_status(
    path: Path,
) -> dict[str, Any]:
    return {
        "path": str(
            path
        ),
        "exists": path.exists(),
        "size_mb": (
            round(
                path.stat().st_size
                / (1024 ** 2),
                3,
            )
            if path.exists()
            else None
        ),
    }


saved_files = [
    CONFIG_DIR
    / "operating_mode_policy_registry.csv",
    CONFIG_DIR
    / "deployment_policy_registry.csv",
    CONFIG_DIR
    / "deployment_config.json",
    TABLE_DIR
    / "operating_mode_commercial_summary.csv",
    TABLE_DIR
    / "operating_mode_daily_portfolio.parquet",
    TABLE_DIR
    / "policy_promotion_gate_evaluation.csv",
    TABLE_DIR
    / "approximate_evidence_horizon.csv",
    TABLE_DIR
    / "operating_mode_fee_spread_sensitivity.csv",
    TABLE_DIR
    / "operating_mode_stress_resilience.csv",
    TABLE_DIR
    / "final_deployment_recommendation.csv",
    MONITORING_DIR
    / "monitoring_rules.csv",
    MONITORING_DIR
    / "fallback_hierarchy.csv",
    MONITORING_DIR
    / "shadow_trading_promotion_plan.csv",
]


verification = {
    "verification_status": (
        verification_status
    ),
    "blocking_failures": (
        blocking_failures
    ),
    "input_manifest": (
        input_manifest
    ),
    "fold_evidence": (
        panel_evidence.to_dict(
            orient="records"
        )
    ),
    "non_reference_production_eligible_policies": (
        production_eligible_non_reference
    ),
    "non_reference_pilot_eligible_policies": (
        pilot_eligible_non_reference
    ),
    "default_live_mode": (
        deployment_config[
            "default_live_mode"
        ]
    ),
    "recommended_d2_pilot_mode": (
        deployment_config[
            "recommended_d2_pilot_mode"
        ]
    ),
    "exact_track_live_status": (
        deployment_config[
            "exact_track_live_status"
        ]
    ),
    "final_recommendations": (
        deployment_recommendation[
            [
                "panel",
                "operating_mode",
                "deployment_status",
                "commercial_delta_vs_conservative_eur",
                "commercial_delta_vs_conservative_eur_per_mwh",
                "area_absolute_imbalance_change_pct",
                "positive_stress_case_share",
                "recommended_use",
            ]
        ]
        .round(6)
        .to_dict(
            orient="records"
        )
    ),
    "saved_files": [
        file_status(
            path
        )
        for path in saved_files
    ],
}


verification_path = (
    AUDIT_DIR
    / "notebook_05_verification.json"
)


with verification_path.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        verification,
        file,
        indent=2,
        default=str,
    )


print(
    "=== NOTEBOOK 05 VERIFICATION ==="
)

print(
    json.dumps(
        verification,
        indent=2,
        default=str,
    )
)


if blocking_failures:
    raise RuntimeError(
        "Notebook 05 verification failed: "
        f"{blocking_failures}"
    )


## Final output directory

Notebook 05 writes to:

```text
outputs/notebook_05_integrated_decision_system/
```

Principal artifacts:

```text
audits/notebook_05_verification.json
audits/panel_evidence_tiers.csv
audits/observed_commercial_fold_lineage.csv

config/operating_mode_policy_registry.csv
config/deployment_policy_registry.csv
config/deployment_config.json

tables/operating_mode_commercial_summary.csv
tables/operating_mode_daily_portfolio.parquet
tables/policy_promotion_gate_evaluation.csv
tables/approximate_evidence_horizon.csv
tables/operating_mode_fee_spread_sensitivity.csv
tables/operating_mode_stress_resilience.csv
tables/final_deployment_recommendation.csv

monitoring/monitoring_rules.csv
monitoring/fallback_hierarchy.csv
monitoring/shadow_trading_promotion_plan.csv
```

Expected final status:

```text
INTEGRATED_DECISION_SYSTEM_COMPLETE
```

The default live policy remains the conservative point nomination. The D2
balanced mode is the recommended controlled pilot. Exact-track alternatives
remain shadow-only until a second independent probabilistic/commercial fold is
available.
